In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import nltk

In [3]:
df=pd.read_csv('emails.csv')

In [4]:
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [5]:
df.head()

,text,spam
0,Subject: naturally irresistible your corporate...,1
1,Subject: the stock trading gunslinger fanny i...,1
2,Subject: unbelievable new homes made easy im ...,1
3,Subject: 4 color printing special request add...,1
4,"Subject: do not have money , get software cds ...",1


In [6]:
df.shape

(5728, 2)

In [9]:
df.isnull().sum()

,0
text,0
spam,0


In [10]:
df.duplicated().sum()

np.int64(33)

In [11]:
df=df.drop_duplicates().reset_index(drop=True)

In [12]:
df.duplicated().sum()

np.int64(0)

In [13]:
df.shape

(5695, 2)

In [14]:
import re
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

def preprocess(text):
  text = str(text).lower()
  text = re.sub(r'[\W_]', ' ', text)
  word_tokens = nltk.word_tokenize(text)
  lemmatizer = WordNetLemmatizer()
  stop_words = set(stopwords.words('english'))
  processed_words = []
  for word in word_tokens:
    if word.isalpha() and word not in stop_words:
      processed_words.append(lemmatizer.lemmatize(word))

  return processed_words

In [19]:
df['processed_text'] = df['text'].apply(preprocess)
df.head()

,text,spam,processed_text
0,Subject: naturally irresistible your corporate...,1,"[subject, naturally, irresistible, corporate, ..."
1,Subject: the stock trading gunslinger fanny i...,1,"[subject, stock, trading, gunslinger, fanny, m..."
2,Subject: unbelievable new homes made easy im ...,1,"[subject, unbelievable, new, home, made, easy,..."
3,Subject: 4 color printing special request add...,1,"[subject, color, printing, special, request, a..."
4,"Subject: do not have money , get software cds ...",1,"[subject, money, get, software, cd, software, ..."


In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Join the list of processed words into a single string for each email
df['processed_text_str'] = df['processed_text'].apply(lambda x: ' '.join(x))

# Initialize TfidfVectorizer
tfidf_vectorizer = TfidfVectorizer(max_features=5000,ngram_range=(1,2),stop_words='english') # You can adjust max_features as needed

# Fit and transform the processed text
X = tfidf_vectorizer.fit_transform(df['processed_text_str'])

print("Shape of TF-IDF matrix:", X.shape)
print("First 5 TF-IDF features for the first email:")
# Convert the sparse matrix row to a dense array to print a few values
print(X[0, :5].toarray())

Shape of TF-IDF matrix: (5695, 5000)
First 5 TF-IDF features for the first email:
[[0. 0. 0. 0. 0.]]


In [23]:
# Separate features (X) and target (y)
y = df['spam']

print("Shape of target variable y:", y.shape)

Shape of target variable y: (5695,)


In [24]:
from sklearn.model_selection import train_test_split

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

print("Shape of X_train:", X_train.shape)
print("Shape of X_test:", X_test.shape)
print("Shape of y_train:", y_train.shape)
print("Shape of y_test:", y_test.shape)

Shape of X_train: (4556, 5000)
Shape of X_test: (1139, 5000)
Shape of y_train: (4556,)
Shape of y_test: (1139,)


In [25]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report

# Initialize the KNN classifier with cosine similarity as the metric
# A smaller 'n_neighbors' (e.g., 5) is a good starting point
knn_classifier = KNeighborsClassifier(n_neighbors=5, metric='cosine')

# Train the classifier
knn_classifier.fit(X_train, y_train)

print("KNN Classifier with Cosine Similarity trained successfully.")

KNN Classifier with Cosine Similarity trained successfully.


In [26]:
y_pred = knn_classifier.predict(X_test)
print("Predictions made on the test set.")

Predictions made on the test set.


In [27]:
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Accuracy: 0.9807

Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       865
           1       0.97      0.95      0.96       274

    accuracy                           0.98      1139
   macro avg       0.98      0.97      0.97      1139
weighted avg       0.98      0.98      0.98      1139



In [28]:
y_train_pred = knn_classifier.predict(X_train)
print("Predictions made on the training set.")

Predictions made on the training set.


In [29]:
accuracy_train = accuracy_score(y_train, y_train_pred)
print(f"Training Accuracy: {accuracy_train:.4f}")

print("\nTraining Classification Report:")
print(classification_report(y_train, y_train_pred))

Training Accuracy: 0.9875

Training Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99      3462
           1       0.98      0.97      0.97      1094

    accuracy                           0.99      4556
   macro avg       0.98      0.98      0.98      4556
weighted avg       0.99      0.99      0.99      4556



In [30]:
from sklearn.ensemble import RandomForestClassifier

# Initialize the Random Forest classifier
# A good starting point for n_estimators is 100, and random_state for reproducibility
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)

# Train the classifier
rf_classifier.fit(X_train, y_train)

print("Random Forest Classifier trained successfully.")

Random Forest Classifier trained successfully.


In [31]:
y_pred_rf = rf_classifier.predict(X_test)
print("Predictions made on the test set using Random Forest.")

Predictions made on the test set using Random Forest.


In [32]:
accuracy_rf = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Test Accuracy: {accuracy_rf:.4f}")

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))

Random Forest Test Accuracy: 0.9851

Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       865
           1       0.98      0.96      0.97       274

    accuracy                           0.99      1139
   macro avg       0.98      0.98      0.98      1139
weighted avg       0.99      0.99      0.99      1139



In [33]:
print("\n--- Model Comparison ---")
print(f"KNN Test Accuracy: {accuracy:.4f}")
print(f"Random Forest Test Accuracy: {accuracy_rf:.4f}")

print("\nKNN Classification Report:")
print(classification_report(y_test, y_pred))

print("\nRandom Forest Classification Report:")
print(classification_report(y_test, y_pred_rf))


--- Model Comparison ---
KNN Test Accuracy: 0.9807
Random Forest Test Accuracy: 0.9851

KNN Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       865
           1       0.97      0.95      0.96       274

    accuracy                           0.98      1139
   macro avg       0.98      0.97      0.97      1139
weighted avg       0.98      0.98      0.98      1139


Random Forest Classification Report:
              precision    recall  f1-score   support

           0       0.99      0.99      0.99       865
           1       0.98      0.96      0.97       274

    accuracy                           0.99      1139
   macro avg       0.98      0.98      0.98      1139
weighted avg       0.99      0.99      0.99      1139



In [36]:
import pickle

# Save the Random Forest model
with open('rf_model.pkl', 'wb') as model_file:
    pickle.dump(rf_classifier, model_file)

# Save the TF-IDF vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as vectorizer_file:
    pickle.dump(tfidf_vectorizer, vectorizer_file)

print('Files saved successfully: rf_model.pkl, tfidf_vectorizer.pkl')

Files saved successfully: rf_model.pkl, tfidf_vectorizer.pkl
